# FedHealth — Notebook 04: Train Model B (beta-thalassemia trait)

**Why this notebook needs a new dataset.** Notebook 01 showed the original `thalassemia_alphanorm.csv` (alpha *silent carriers*) is not separable from CBC (AUC ~0.5) — alpha carriers are largely invisible on a blood count. **Beta-thalassemia trait**, by contrast, is genuinely CBC-detectable (low MCV/MCH, high RBC, Mentzer < 13) and is a well-studied ML target. So Model B is a beta-thal-trait screener; you supply the dataset (see below).

This is the ML half of the thal strategy; the interpretable rule layer (`fedhealth_ml.rules`, used in Notebook 03) is the other half and needs no new data.

## Dataset options (pick one, download, drop the CSV into Colab)
Public CBC datasets for beta-thal trait vs iron deficiency / normal:

- **NIBD Karachi (301 CBC reports)** — *Pakistani data*, best fit for the scope's 'locally representative' claim. Described in the PLOS One 2025 paper 'From CBC to clarity'; may require requesting from the authors.
- **Mendeley — Iraq thalassemia (1073 patients, CBC + HPLC):** https://data.mendeley.com/datasets/p8rv84hrbs/1
- **Mendeley — Anemia CBC, Lucknow India (has RDW, RBC, WBC, PLT):** https://data.mendeley.com/datasets/dy9mfjchm7/1
- **Mendeley — Thalassemia clinical, Bangladesh (617, RBC indices + HbF):** https://data.mendeley.com/datasets/8kcdkxmcjw/1
- **Kaggle** — search 'thalassemia CBC' (e.g. a 616-record set with 13 CBC features incl. RDW).

Prefer a dataset that includes **RDW** (unlocks the Green-King / RDW-index discriminants) and, ideally, an **iron-deficiency** class so the model learns the clinically hard IDA-vs-BTT boundary, not just BTT-vs-healthy.

In [ ]:
# --- Setup ------------------------------------------------------------------
# Module 3 Colab setup.
#
# Prerequisite:
# Clone the GitHub testing branch into /content/FedHealth-FYP before running
# these notebooks. Dataset CSVs remain in Google Drive at MyDrive/FYP.
#
# This repository stores shared files flat in "Module 3 ML/". The code below
# creates the temporary "fedhealth_ml" package layout expected by the notebooks.

from pathlib import Path
from google.colab import drive
import os
import shutil
import sys

REPO_DIR = Path('/content/FedHealth-FYP')
SOURCE_DIR = REPO_DIR / 'Module 3 ML'
assert SOURCE_DIR.exists(), (
    'Repository files are missing. Clone the GitHub testing branch first.'
)

if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

!pip -q install -r "/content/FedHealth-FYP/Module 3 ML/requirements.txt"

RUNTIME_DIR = Path('/content/fedhealth_runtime')
PACKAGE_DIR = RUNTIME_DIR / 'fedhealth_ml'
PACKAGE_DIR.mkdir(parents=True, exist_ok=True)

for filename in ['schema.py', 'preprocess.py', 'models.py', 'rules.py']:
    shutil.copy2(SOURCE_DIR / filename, PACKAGE_DIR / filename)

(PACKAGE_DIR / '__init__.py').write_text(
    '# Temporary Colab package for Module 3 validation.\n'
)

if str(RUNTIME_DIR) not in sys.path:
    sys.path.insert(0, str(RUNTIME_DIR))

DATA_DIR = '/content/drive/MyDrive/FYP'
ANEMIA = f'{DATA_DIR}/anemia_diagnosed_cbc_data_v4.csv'
THAL   = f'{DATA_DIR}/thalassemia_alphanorm.csv'
LEUK   = f'{DATA_DIR}/biased_leukemia_dataset.csv'

for dataset_path in [ANEMIA, THAL, LEUK]:
    assert os.path.exists(dataset_path), f'Dataset not found: {dataset_path}'

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from fedhealth_ml import schema, preprocess as pp

print('fedhealth_ml loaded. Model A features:', len(schema.MODEL_A_FEATURES))
print('All three datasets found in Google Drive.')


## 1. Configure for YOUR dataset
Set the path, the label column, and the label map to match the file you downloaded. Class keys must come from `schema.MODEL_B_CLASSES` (drop 'iron_deficiency' if your dataset is binary).

In [ ]:
from fedhealth_ml import schema, preprocess as pp, models, rules

THAL_CSV  = f'{DATA_DIR}/beta_thal_cbc.csv'   # <-- your downloaded file
LABEL_COL = 'phenotype'                        # <-- your label column
# Map your dataset's raw label strings -> canonical class keys.
LABEL_MAP = {
    'normal': 'normal', 'control': 'normal',
    'iron deficiency anemia': 'iron_deficiency', 'ida': 'iron_deficiency',
    'beta thalassemia trait': 'beta_thal_trait', 'btt': 'beta_thal_trait',
    'beta thalassemia minor': 'beta_thal_trait',
}
# Feature columns present in your file (subset of schema.MODEL_B_FEATURES).
FEATURES = [c for c in schema.MODEL_B_FEATURES]  # adjust if names differ


## 2. Load + inspect class balance

In [ ]:
X, y = pp.load_thal(THAL_CSV, label_col=LABEL_COL, label_map=LABEL_MAP, features=FEATURES)
classes = [c for c in schema.MODEL_B_CLASSES if c in set(y)]
print('rows', len(X), '| features', list(X.columns))
print('class balance:', y.value_counts().to_dict())

## 3. GUARD: confirm the dataset actually has CBC signal
Learn from the alpha lesson — before training, verify separability. If the cross-validated AUC/macro-F1 is near chance, STOP and pick a better dataset rather than training a noise model.

In [ ]:
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
yi = pp.encode_labels(y, classes)
cv = StratifiedKFold(5, shuffle=True, random_state=42)
pipe = make_pipeline(SimpleImputer(strategy='median'), StandardScaler(),
                     LogisticRegression(max_iter=2000, class_weight='balanced'))
pred = cross_val_predict(pipe, X, yi, cv=cv)
mf1 = f1_score(yi, pred, average='macro')
print(f'5-fold macro-F1 = {mf1:.3f}')
assert mf1 > 0.6, 'Low signal (<0.6). Reconsider the dataset before training.'

## 4. Feature engineering: add discriminant indices (best of A + B)
We append the Mentzer/discriminant index VALUES as extra model features, so the ANN+XGBoost hybrid benefits from the same clinical knowledge the rule layer encodes.

In [ ]:
import numpy as np, pandas as pd
def add_indices(df):
    out = df.copy()
    recs = df.to_dict('records')
    idx = [rules.thalassemia_trait_flag(r)['indices'] for r in recs]
    for name in ['mentzer','england_fraser','shine_lal','srivastava','ehsani','sirdah']:
        out[f'idx_{name}'] = [d.get(name, {}).get('value', np.nan) for d in idx]
    return out
Xe = add_indices(X)
ENG_FEATURES = list(Xe.columns)
print('engineered feature count:', len(ENG_FEATURES))

## 5. Preprocess (fit on train only) + split

In [ ]:
sp = pp.stratified_split(Xe, y, test_size=0.2, val_size=0.1, seed=42)
import os; ART='/content/artifacts/modelB'; os.makedirs(ART, exist_ok=True)
prep = pp.CBCPreprocessor(ENG_FEATURES).fit(sp['X_train'])
prep.save(f'{ART}/preprocessor.joblib')
Xtr, Xval, Xte = (prep.transform(sp[k]) for k in ['X_train','X_val','X_test'])
ytr = pp.encode_labels(sp['y_train'], classes)
yval = pp.encode_labels(sp['y_val'], classes)
yte  = pp.encode_labels(sp['y_test'], classes)
print('train', Xtr.shape, '| classes', classes)

## 6. Train the same Hybrid ANN -> XGBoost (small net for small data)

In [ ]:
n = len(classes)
ann = models.build_ann(input_dim=Xtr.shape[1], n_classes=n, hidden=(32,16), dropout=0.3)
cw = models.class_weights(ytr, n)
models.train_ann(ann, Xtr, ytr, Xval, yval, class_weight=cw,
                 epochs=200, batch_size=16, patience=25, verbose=0)
hybrid = models.HybridModel(ann, classes, model_name='thal_screen', version='v1.0.0')
hybrid.fit_meta(Xtr, ytr, Xval, yval, n_estimators=200, max_depth=3)
print('trained. temperature T =', round(hybrid.temperature.T, 3))

## 7. Evaluate + save

In [ ]:
proba = hybrid.predict_proba(Xte); pred = proba.argmax(1)
metrics = models.compute_metrics(yte, pred, classes)
print('macro-F1:', metrics['macro_f1'], '| balanced-acc:', metrics['balanced_accuracy'])
for c in classes:
    pc = metrics['per_class'][c]; print(f'  {c:18} recall={pc["recall"]:.2f} support={pc["support"]}')
hybrid.save(ART); models.write_registry_metrics(ART, metrics)
print('saved Model B artifacts ->', ART)

## 8. Federation + registry
Model B federates exactly like Model A (`models.get_ann_weights` -> Flower). Register it as `thal_screen` in the ml_models table (Module 6 route). Note for defense: the standalone `rules.thalassemia_trait_flag` gives an interpretable second opinion that doesn't depend on this trained model.